In [ ]:
# Milestone 3 setup (run first, before answering any questions)
# Run this code in a code cell before answering any of the question. This will create your knowledge 
# base and the FAISS index for this milestone.

%pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 

import faiss 

from sentence_transformers import SentenceTransformer, CrossEncoder 

from transformers import AutoTokenizer, pipeline 

from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('../../data/train.csv') 

print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 

model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 65.1 MB/s eta 0:00:00:00:0100:01
Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [2]:
# Zero-shot classifier for Q1, Q2, Q6

zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 

row_150 = train.iloc[150] 

prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 

ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
# Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the 
# row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted 
# probability score assigned to the ground-truth correct option (option in the 
# answer column)? (Round to 3 decimal points)

res_q1 = zs(prompt_150, candidate_labels=labels_150)

idx = res_q1['labels'].index(ans_150)

score_q1 = res_q1['scores'][idx]
print(f"Q1 Answer: {score_q1:.3f}")

Q1 Answer: 0.384


In [4]:
# Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your 
# FAISS index to retrieve the top k=10 most similar documents. At what exact rank 
# (1 through 10) did FAISS place the true correct document (which is the document 
# originally located at index 150 in the KB)?

prompt_150_emb = model.encode([prompt_150])

k = 10
distances, indices = index.search(prompt_150_emb, k)
retrieved_indices = indices[0]

target_index = 150

rank_q2 = np.where(retrieved_indices == target_index)[0][0] + 1
print(f"Q2 Answer: Rank {rank_q2}")

Q2 Answer: Rank 10


In [5]:
# Code to use a Cross-Encoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices] #Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [6]:
# Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load 
# cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents 
# and sort them by the cross-encoder's score. At what exact rank (1 through 10) 
# does the Cross-Encoder place the true correct document?

reranked_order = np.argsort(ce_scores)[::-1]
reranked_indices = retrieved_indices[reranked_order]

rank_q3 = np.where(reranked_indices == target_index)[0][0] + 1
print(f"Q3 Answer: Rank {rank_q3}")

Q3 Answer: Rank 1


In [7]:
# Q4. Retrieve the top k=5 documents for the prompt at row index 42. 
# Concatenate them with a single space between each. Create a string: 
# "Context: [concatenated_docs] Question: [prompt]". Tokenize this string 
# using the bert-base-uncased tokenizer (without truncation). Exactly how 
# many total tokens does this generate?

row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])
prompt_42_emb = model.encode([prompt_42])

distances_42, indices_42 = index.search(prompt_42_emb, 5)
retrieved_docs_42 = [kb[i] for i in indices_42[0]]

concatenated_docs = " ".join(retrieved_docs_42)

rag_string_42 = f"Context: {concatenated_docs} Question: {prompt_42}"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens_42 = tokenizer(rag_string_42, truncation=False)

total_tokens = len(tokens_42['input_ids'])
print(f"Q4 Answer: {total_tokens}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q4 Answer: 216


In [8]:
# Q5. Retrieve the exact true document for row index 150 from your KB. 
# Create a RAG string: "Context: [true_document] Question: [prompt]". 
# Run the same zero-shot classification from Question 1 on this augmented string. 
# What is the new predicted probability score of the ground-truth correct option? 
# (Round to 3 decimal places).

true_document_150 = kb[150] 

rag_string_150 = f"Context: {true_document_150} Question: {prompt_150}"

res_q5 = zs(rag_string_150, candidate_labels=labels_150)

idx_q5 = res_q5['labels'].index(ans_150)
score_q5 = res_q5['scores'][idx_q5]

print(f"Q5 Answer: {round(score_q5, 3)}")

Q5 Answer: 0.989


In [9]:
# Q6. What happens if your vector database retrieves the wrong information? 
# Take the prompt for row index 150. Manually force the context to be the document 
# located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier 
# on this "Adversarial RAG" string. What is the probability of the correct option now? 
# (Round to 3 decimal places).

bad_document = kb[999]

adv_rag_string = f"Context: {bad_document} Question: {prompt_150}"

res_q6 = zs(adv_rag_string, candidate_labels=labels_150)

idx_q6 = res_q6['labels'].index(ans_150)
score_q6 = res_q6['scores'][idx_q6]

print(f"Q6 Answer: {score_q6:.3f}")

Q6 Answer: 0.529


In [10]:
# In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

# Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. 
# If the exact string of the row's correct option is found inside any of those 5 retrieved documents, 
# it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? 
# (Round to 1 decimal place).

hits = 0
n_rows = 100

for i in range(n_rows):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    correct_letter = row['answer']
    correct_text = str(row[correct_letter])
    
    prompt_emb = model.encode([prompt], show_progress_bar=False)
    distances, indices = index.search(prompt_emb, 5)
    retrieved_docs = [kb[idx] for idx in indices[0]]
    
    hit = any(correct_text in doc for doc in retrieved_docs)
    
    if hit:
        hits += 1

hit_rate = (hits / n_rows) * 100
print(f"Q7 Answer: {hit_rate:.1f}")

Q7 Answer: 73.0


In [11]:
# Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.

# For each row, your pipeline must do the following in order:

# Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

# Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. 
# Select the single document with the highest cross-encoder score.

# Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

# Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 
# 5 options (A, B, C, D, E) as your candidate_labels.

# Score: Look at the probability scores output by the model. Rank the options from 
# highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate 
# the MAP@3 for that row.

# What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? 
# (Round to 3 decimal places).

map3_scores = []
n_rows_q8 = 20

for i in range(n_rows_q8):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    ans_letter = row['answer']
    ans_text = str(row[ans_letter])
    
    labels = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
    
    prompt_emb = model.encode([prompt], show_progress_bar=False)
    distances, indices = index.search(prompt_emb, 5)
    retrieved_docs = [kb[idx] for idx in indices[0]]
    
    pairs = [[prompt, doc] for doc in retrieved_docs]
    ce_scores = cross_encoder.predict(pairs)

    best_doc_idx = np.argmax(ce_scores) 
    best_document = retrieved_docs[best_doc_idx]
    
    rag_string = f"Context: {best_document} Question: {prompt}"
    
    res = zs(rag_string, candidate_labels=labels)
    
    rank = res['labels'].index(ans_text) + 1
    
    if rank == 1:
        ap = 1.0
    elif rank == 2:
        ap = 0.5
    elif rank == 3:
        ap = 1.0 / 3.0
    else:
        ap = 0.0
        
    map3_scores.append(ap)

final_map3 = np.mean(map3_scores)
print(f"Q8 Answer: {final_map3:.3f}")

Q8 Answer: 0.975
